# CD CosMx WTX biopsies: processing

Two WTX slides (I0476 and I0368), each holding biopsies of two patients (inflamed I0476 and non-inflamed I0453; inflamed I0368 and control I0477).

1. Read the AtoMx flat files, assign FOVs to tissue sections (`slide`), keep raw counts in `layers['raw']`.
2. Remove negative and system-control probes; QC (counts, genes, cell area); merge the slides; total-count normalisation and log1p.
3. ComBat by section for the embedding (PCA, neighbours, UMAP, Leiden); `.X` keeps the log-normalised values.
4. Export total-count-normalised counts for AUCell (`count_csv/norm_counts.csv`); read the AUCell labels (`anno/norm_counts_aucell.csv`, shared 50-type gene sets) and map them to cell type, category and general type.
5. Add patient and condition per section and write the annotated object.

**Output:** `Processed_merged/h5ad_obj/norm_anno.h5ad` (input of `02`), per-slide objects.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy.external as sce

In [ ]:
sample_dir = '/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/3 Raw data/R6221_I0476/'

In [ ]:
fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
if 'FOV' in fov_df.columns:
  print("Renaming column 'FOV' to 'fov'.")
  # Rename 'FOV' column to 'fov'
  fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
  # have fov_file reference the new, formatted file and write it
  fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
  fov_df.to_csv(fov_file, index=False)

In [ ]:
adata = sq.read.nanostring(
    path=sample_dir,
    counts_file="wtx_I0476_exprMat_file.csv.gz",
    meta_file="wtx_I0476_metadata_file.csv.gz",
    fov_file="fov_positions_formatted.csv",
)

In [ ]:
adata

In [ ]:
sample_dir = '/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/3 Raw data/R6221_I0368/'

In [ ]:
fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
if 'FOV' in fov_df.columns:
  print("Renaming column 'FOV' to 'fov'.")
  # Rename 'FOV' column to 'fov'
  fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
  # have fov_file reference the new, formatted file and write it
  fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
  fov_df.to_csv(fov_file, index=False)

In [ ]:
adata2 = sq.read.nanostring(
    path=sample_dir,
    counts_file="wtx_I0368_exprMat_file.csv.gz",
    meta_file="wtx_I0368_metadata_file.csv.gz",
    fov_file="fov_positions_formatted.csv",
)


In [ ]:
adata2

In [ ]:
adata.var['features']=adata.var.index.tolist()
adata.obs[['orig.ident']]='i0476'
adata.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata.obs['fov'].tolist(), adata.obs['cell_id'].tolist())]

In [ ]:
adata2.var['features']=adata2.var.index.tolist()
adata2.obs[['orig.ident']]='i0368'
adata2.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata2.obs['fov'].tolist(), adata2.obs['cell_id'].tolist())]

In [ ]:
adata.layers['raw']=adata.X

In [ ]:
adata2.layers['raw']=adata2.X

In [ ]:
slide=[]
slide1_list = [1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 17, 18, 19, 20, 23, 24, 25, 26, 29, 30, 31]

slide2_list = [8, 9, 14, 15, 16, 21, 22, 27, 28, 32, 33, 34]

slide3_list = [37, 38, 43, 44, 50, 51, 57, 58, 64, 65, 70, 71, 77, 78, 79]

slide4_list = [35, 36, 39, 40, 41, 42, 45, 46, 47, 52, 53, 54, 59, 60, 61, 66, 67, 72, 73]

for i,j in zip(adata.obs['orig.ident'].tolist(),adata.obs['fov'].tolist()):
    j= int(j)
    if j in slide1_list:
        slide.append(i+'_'+'slide1')
    elif j in slide2_list:
        slide.append(i+'_'+'slide2')
    elif j in slide3_list:
        slide.append(i+'_'+'slide3')
    elif j in slide4_list:
        slide.append(i+'_'+'slide4')
    else:
        slide.append(i+'_'+'slide5')

In [ ]:
adata.obs['slide']=slide

In [ ]:
slide=[]
slide1_list = list(range(17))                      # [0, 1, 2, ..., 16]
slide2_list = [17, 18, 19, 20, 21]
slide3_list = list(range(22, 51)) + [70]           # [22..50] + 70
slide4_list = [51, 52, 53, 69]
slide5_list = list(range(54, 71))                  # [54..70]

for i,j in zip(adata2.obs['orig.ident'].tolist(),adata2.obs['fov'].tolist()):
    j= int(j)
    if j in slide1_list:
        slide.append(i+'_'+'slide1')
    elif j in slide2_list:
        slide.append(i+'_'+'slide2')
    elif j in slide3_list:
        slide.append(i+'_'+'slide3')
    elif j in slide4_list:
        slide.append(i+'_'+'slide4')
    elif j in slide5_list:
        slide.append(i+'_'+'slide5')
    else:
        print(j,'error')

In [ ]:
adata2.obs['slide']=slide

In [ ]:
adata.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0476/h5ad_obj/full_noqc.h5ad')
adata2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0368/h5ad_obj/full_noqc.h5ad')

In [ ]:
adata=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0476/h5ad_obj/full_noqc.h5ad')
adata2=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0368/h5ad_obj/full_noqc.h5ad')

In [ ]:
adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
adata.var['control_probe']=[True if 'SystemContro' in i else False for i in adata.var.index.tolist()]
adata.var

In [ ]:
adata2.var['neg_probe']=[True if 'Negative' in i else False for i in adata2.var.index.tolist()]
adata2.var['control_probe']=[True if 'SystemContro' in i else False for i in adata2.var.index.tolist()]
adata2.var

In [ ]:
adata=adata[:, ~adata.var["neg_probe"]]
adata=adata[:, ~adata.var["control_probe"]]
adata2=adata2[:, ~adata2.var["neg_probe"]]
adata2=adata2[:, ~adata2.var["control_probe"]]

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, inplace=True, log1p=True
)
sc.pp.calculate_qc_metrics(
    adata2, inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata2,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=400)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_cells(adata, max_counts=10000)
sc.pp.filter_cells(adata, max_genes=6000)

In [ ]:
adata=adata[adata.obs['Area']<=40000]

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata2, min_counts=400)
sc.pp.filter_cells(adata2, min_genes=200)
sc.pp.filter_cells(adata2, max_counts=10000)
sc.pp.filter_cells(adata2, max_genes=6000)

In [ ]:
adata2=adata2[adata2.obs['Area']<=40000]

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata2,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_cosmx=sc.concat([adata, adata2])

In [ ]:
sc.pp.normalize_total(adata_cosmx, inplace=True)
sc.pp.log1p(adata_cosmx)

In [ ]:
adata.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0476/h5ad_obj/qc.h5ad')
adata2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0368/h5ad_obj/qc.h5ad')

In [ ]:
adata_cosmx.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/qc_norm.h5ad')

In [ ]:
adata=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0476/h5ad_obj/qc.h5ad')
adata2=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0368/h5ad_obj/qc.h5ad')

In [ ]:
adata_cosmx=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/qc_norm.h5ad')

In [ ]:
norm_log = adata_cosmx.X.copy()

In [ ]:
sc.pp.combat(adata_cosmx, key="slide") 

In [ ]:
sc.tl.pca(adata_cosmx, n_comps=50, svd_solver="arpack")  # avoids full dense mem usage


In [ ]:
adata_cosmx.layers['combat']=adata_cosmx.X

In [ ]:
adata_cosmx.X=norm_log

In [ ]:
sc.pp.neighbors(adata_cosmx,n_neighbors=30,metric='cosine',use_rep='X_pca')
sc.tl.umap(adata_cosmx, min_dist=0.01, spread=5)
sc.tl.leiden(adata_cosmx)

In [ ]:
adata_cosmx.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/qc_norm_umap.h5ad')

In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='i0476']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='i0368']

In [ ]:
adata_cosmx1.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0476/h5ad_obj/qc.h5ad')
adata_cosmx2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0368/h5ad_obj/qc.h5ad')

In [ ]:
adata_cosmx_norm_nolog1p = adata_cosmx.copy()
adata_cosmx_norm_nolog1p.X = adata_cosmx_norm_nolog1p.layers['raw']


In [ ]:
adata_cosmx_norm_nolog1p.X.toarray()

In [ ]:
sc.pp.normalize_total(adata_cosmx_norm_nolog1p, inplace=True)

In [ ]:
adata_cosmx_norm_nolog1p_count = pd.DataFrame(adata_cosmx_norm_nolog1p.X.toarray())

In [ ]:
adata_cosmx_norm_nolog1p_count

In [ ]:
adata_cosmx_norm_nolog1p_count.columns = adata_cosmx_norm_nolog1p.var.index.tolist()
adata_cosmx_norm_nolog1p_count.index=adata_cosmx_norm_nolog1p.obs['cell'].tolist()

In [ ]:
adata_cosmx_norm_nolog1p_count

In [ ]:
adata_cosmx_norm_nolog1p_count.to_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/count_csv/norm_counts.csv')

In [ ]:
adata_cosmx=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/qc_norm_umap.h5ad')

In [ ]:
adata_cosmx

# anno

In [ ]:
anno = pd.read_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/anno/norm_counts_aucell.csv')


In [ ]:
anno

In [ ]:
anno['label'].value_counts()

In [ ]:
adata_cosmx.obs['fov_cell_id']

In [ ]:
mapper= pd.read_csv('/share/fsmresfiles/UC/scRNA-seq/merged_cd/cell_type_category_map.csv')

In [ ]:
mapper

In [ ]:
# Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
mapper

In [ ]:
mapper['cell_type_aucell'].unique()

In [ ]:
anno['label_clean'] = anno['label'].str.rstrip()

In [ ]:
anno['label_clean'].unique()

In [ ]:
anno_map=anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
anno_map.isna().sum()

In [ ]:
anno_map

In [ ]:
adata_cosmx.obs['cell_type'] = anno_map['cell type'].tolist()
adata_cosmx.obs['cell_type_short'] = anno_map['cell type short'].tolist()
adata_cosmx.obs['cell_category'] = anno_map['cell category'].tolist()
adata_cosmx.obs['cell_type_general'] = anno_map['cell type general'].tolist()

In [ ]:
base_cats = list(adata_cosmx.obs['cell_category'].astype('category').cat.categories)
reordered_cats = [cat for cat in base_cats if cat != 'Cycling'] + ['Cycling']

adata_cosmx.obs['cell_category'] = pd.Categorical(
    adata_cosmx.obs['cell_category'],
    categories=reordered_cats,
    ordered=True
)

In [ ]:
adata_cosmx.obs['slide'].unique()

In [ ]:
patient =[]
condition = []

for i,j in adata_cosmx.obs.iterrows():
    if j['orig.ident'] == 'i0476':
        if j['slide'] in ['i0476_slide1','i0476_slide2']:
            patient.append('I0476')
            condition.append('Inf')
        else:
            patient.append('I0453')
            condition.append('Non-Inf')
    else:
        if j['slide'] in ['i0368_slide1','i0368_slide2']:
            patient.append('I0368')
            condition.append('Inf')
        else:
            patient.append('I0477')
            condition.append('Healthy')

In [ ]:
adata_cosmx.obs['patient'] = patient
adata_cosmx.obs['condition'] = condition

In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='i0476']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='i0368']

In [ ]:
adata_cosmx.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata_cosmx1.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0476/h5ad_obj/norm_anno.h5ad')
adata_cosmx2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_I0368/h5ad_obj/norm_anno.h5ad')